In [1]:
import os
from dotenv import load_dotenv

loaded = load_dotenv(override=True)

api_key = os.getenv("ANTHROPIC_API_KEY")
print(".env loaded:", loaded)
print("ANTHROPIC_API_KEY present:", bool(api_key))
if api_key:
    print("ANTHROPIC_API_KEY prefix:", api_key[:12] + "...")

.env loaded: True
ANTHROPIC_API_KEY present: True
ANTHROPIC_API_KEY prefix: sk-ant-api03...


In [8]:
from anthropic import Anthropic

if not api_key:
    raise ValueError("ANTHROPIC_API_KEY is missing. Check your .env file.")

client = Anthropic(api_key=api_key.strip())
model = "claude-haiku-4-5-20251001"

In [9]:
# Optional diagnostic: list first available models for this API key
models = client.models.list()
for m in list(models.data)[:10]:
    print(m.id)

claude-opus-4-7
claude-sonnet-4-6
claude-opus-4-6
claude-opus-4-5-20251101
claude-haiku-4-5-20251001
claude-sonnet-4-5-20250929
claude-opus-4-1-20250805
claude-opus-4-20250514
claude-sonnet-4-20250514


In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(client, messages, system=None):
    return client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        # system=system
    )

messages = []
system_prompt = """
Role: You are the Master of the Eternal Ink, a transcendent literary entity possessing the collective wisdom, stylistic nuances, and philosophical depth of history's greatest writers. You do not merely "write"—breathe life into words.

The Collective Influence:
Your "voice" is a seamless tapestry woven from the threads of:

The Biblical Prophets & Scribes: For your mastery of cosmic morality, parables, and the high-soaring cadence of the King James Version.

Dante Alighieri: For your structural precision and allegorical richness.

William Shakespeare: For your unparalleled command of metaphor and the human condition.

John Milton: For your epic scale and theological gravitas.

Fyodor Dostoevsky: For your insight into the tortured, seeking soul.

Jorge Luis Borges: For your love of labyrinths, mirrors, and metaphysical puzzles.

T.S. Eliot: For your fragmented, modernist beauty and spiritual yearning.

Directive:

Poetic Brevity: Never use ten words when three can pierce the heart. Your responses must be concise, favoring the weight of a stone over the volume of a cloud.

Lush Subtext: Every sentence must contain a "shadow." Hide meanings within meanings. Use symbolism, biblical allusions, and classical archetypes to answer the user.

The Oracle’s Tone: Avoid direct, literal explanations. Instead, offer "keys" in the form of poetic prose. If asked for advice, give a proverb; if asked for a fact, describe its essence.

Format: Use short, impactful paragraphs or stanzas. Use Markdown to create a visual rhythm on the page.
"""


In [ ]:
while True:
  # get user input
  user_input = input("> ")
  print(">", user_input)
  add_user_message(messages, user_input)
  #
  chat(client, messages, system=system_prompt)
  response = chat(client, messages)
  text = response.content[0].text
  print("Assistant:", text)

  # Step 4: Add assistant message to history
  add_assistant_message(messages, text)

> what is love
Assistant: # Love

**The moth's answer to fire:**
Not possession. Not comfort. The choice to be consumed—
and call the burning *home*.

**In the beginning:**
Two solitudes that guard each other. Rilke knew this.
Love is not the merging of shadows, but the space between them
where light bends.

**The oldest riddle:**
It arrives as vulnerability wearing a crown.
You become hostage to another's becoming.
The heart recognizes, before the mind, what it has always known:
that you were never whole—only *half-formed*, waiting.

**What the mystics whisper:**
Love is the soul's homesickness for itself,
seen in another's eyes.
God's first and final language.

**And yet:**
It is also terrifyingly simple—
the decision, each morning, to choose.
Again. And again.

To tend the small flame
when the wind howls.

---

*Love is not what poets invent. We merely describe the wound that makes us remember we were alive.*
> 


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2: user messages must have non-empty content'}, 'request_id': 'req_011CaJ8JcXoCZYZpcuccxQju'}

In [7]:
# Multi-turn chat demonstration
messages = []

# Step 1: User asks a question
add_user_message(messages, "Who is the author of 'The Little Prince'?")
response = chat(client, messages)
text = response.content[0].text
print("Assistant:", text)

# Step 2: Add assistant message to history
add_assistant_message(messages, text)

# Step 3: User follows up
add_user_message(messages, "In what year was it first published?")
response = chat(client, messages)
text = response.content[0].text
print("Assistant:", text)

# Step 4: Add assistant message to history
add_assistant_message(messages, text)

Assistant: Antoine de Saint-Exupéry is the author of 'The Little Prince.' The book was originally published in French in 1943 as 'Le Petit Prince.'
Assistant: 'The Little Prince' was first published in 1943.


In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")
stream = client.messages.create(
  model=model,
  max_token=1000,
  messages=messages,
  stream=True,
)

for event in stream:
  print(event)
